# Setup

In [ ]:
!git clone ANONYMIZED_REPO_URL (I will make this public this later)

Cloning into 'RL_Signaling'...
remote: Enumerating objects: 1081, done.
remote: Counting objects: 100% (171/171), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 1081 (delta 67), reused 117 (delta 19), pack-reused 910 (from 2)
Receiving objects: 100% (1081/1081), 116.61 MiB | 47.78 MiB/s, done.
Resolving deltas: 100% (592/592), done.


In [11]:
%cd RL_Signaling

/content/RL_Signaling/RL_Signaling


In [12]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import random

import networkx as nx
import numpy as np
import pandas as pd
from tqdm import tqdm

from rl_signaling.agents import QLearningAgent, TDLearningAgent, UrnAgent
from rl_signaling.env import NetMultiAgentEnv, TempNetMultiAgentEnv
from rl_signaling.games import create_random_canonical_game
from rl_signaling.simulation import simulation_function, temp_simulation_function


In [13]:
# Decide where to put the files and do the working
from google.colab import drive
drive.mount('/content/drive')

dump_path = '/content/drive/My Drive/Colab Projects/Python ABMs/Communication/Plots and Datasets/'
print("Current Directory:", dump_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current Directory: /content/drive/My Drive/Colab Projects/Python ABMs/Communication/Plots and Datasets/


# Canonical Model with Signal Costs

- World States: Two binary variables X, Y
- agents_observed_variables = {0:[0],1:[1]}
- Random Canonical Games
- n_features = 2
- n_signaling_actions = 2
- n_final_actions = 4
- Signal Cost in [0.1,0.5]

In [14]:
# add_data = False
# n_iterations = 10

## Urn Agent (retired)

The Roth-Erev costly-signaling block was retired on 2026-05-09. Costly signaling is theoretically ill-defined under the project's non-negativity-clamped Roth-Erev urn rule:

1. **Negative net rewards.** When `game_reward < cost` and a non-null signal is emitted, the net reward is negative. The clamp `urn[a] = max(0, urn[a] + r)` introduces an absorbing barrier — once `urn[a] = 0` the action can never recover.
2. **Real-valued reinforcement.** The classical Roth-Erev urn was specified for non-negative integer reinforcement; the costly net reward is real-valued, breaking the urn-as-counter interpretation.

See [`analytics/agent_urn.md`](../analytics/agent_urn.md) (Applicability constraints) and [`analytics/costly_signaling.md`](../analytics/costly_signaling.md) (Compatibility with the project's three agent types) for the formal treatment. Q-learning's TD update is defined on the reals and handles costly rewards natively, so the Q-Learning costly block below is unaffected.


## Q-Learning

In [17]:
simulate=True
if simulate:
  add_data = False
  n_iterations = 10000

  # Print number of available CPU cores
  n_cores = cpu_count()
  print(f"Using all available CPU cores: {n_cores}")

  # Define column names
  column_names = [
      'iteration', 'n_signaling_actions', 'n_final_actions', 'full_information', 'with_signals', 'Signal_Cost_A0', 'Signal_Cost_A1',
      'Agent_0_Initial_NMI', 'Agent_0_NMI', 'Agent_0_avg_reward', 'Agent_0_final_reward',
      'Agent_1_Initial_NMI', 'Agent_1_NMI', 'Agent_1_avg_reward', 'Agent_1_final_reward'
  ]

  n_episodes = 10000
  n_agents = 2
  n_features = 2
  n_signaling_actions = 2
  n_final_actions = 4

  def run_single_case(iteration, full_info, with_signals, signal_cost, game_dicts, obs_vars, graph):
          #set seeds
      np.random.seed(iteration)
      random.seed(iteration)
      # continue

      env = NetMultiAgentEnv(n_agents=n_agents, n_features=n_features,
                      n_signaling_actions=n_signaling_actions,
                      n_final_actions=n_final_actions,
                      full_information = full_info,
                      game_dicts=game_dicts,
                      observed_variables = obs_vars,
                      agent_type=QLearningAgent,
                      initialize = False,
                      costly_signaling=True,
                      graph=graph)

      effective_n_signaling_actions = n_signaling_actions + 1
      env.agents = [
              QLearningAgent(
                  n_signaling_actions=effective_n_signaling_actions,
                  n_final_actions=n_final_actions,
                  exploration_rate=0.9652628633727897,
                  exploration_decay=0.9998122815486062,
                  min_exploration_rate=1e-10,
                  choice='ucb',
                  exp_smoothing=False,
                  costly_signaling=True
              ) for _ in range(n_agents)
          ]


      results = [iteration, n_signaling_actions, n_final_actions, full_info, with_signals,signal_cost[0],signal_cost[1]]

      signal_usage, rewards_history, signal_information_history, nature_history, histories = simulation_function(n_agents=n_agents,
                      n_features=n_features, n_signaling_actions=n_signaling_actions,
                      n_final_actions=n_final_actions,
                      n_episodes=10000, with_signals = with_signals,plot=True,env=env, verbose=False,
                      signal_cost = signal_cost,
                      costly_signaling=True)

      for agent_id in range(n_agents):
          info_hist = signal_information_history[agent_id]
          reward_hist = rewards_history[agent_id]
          results.extend([
              np.mean(info_hist[:10]),
              np.mean(info_hist[-100:]),
              np.mean(reward_hist),
              np.mean(reward_hist[-100:])
          ])

      return results

  def run_all_cases_for_iteration(iteration):
      # Prepare shared data for all 4 cases
      game_dicts = {i: create_random_canonical_game(n_features, n_final_actions) for i in range(n_agents)}
      obs_vars = {0: [0], 1: [1]}
      rdn = np.random.uniform(0.0, 0.5)
      signal_cost = [rdn,rdn]
      G = nx.DiGraph()
      G.add_edges_from([(0, 1), (1, 0)])

      cases = [(False, True)]
      return [run_single_case(iteration, fi, ws, signal_cost,game_dicts, obs_vars, G) for fi, ws in cases]

  # Run simulations in parallel using all available cores
  all_results = Parallel(n_jobs=n_cores)(
      delayed(run_all_cases_for_iteration)(i) for i in tqdm(range(n_iterations), desc="Running in parallel")
  )

  # Flatten the list of lists
  flat_results = [row for group in all_results for row in group]
  results_df = pd.DataFrame(flat_results, columns=column_names)

  # Append or save
  output_file = dump_path+'qlearning_results_canonical_costly_signal.csv'
  if add_data:
      old_results_df = pd.read_csv(output_file)
      total_results_df = pd.concat([old_results_df, results_df], ignore_index=True)
  else:
      total_results_df = results_df

  total_results_df.to_csv(output_file, index=False)
  print(f"Total rows: {len(total_results_df)}")

Using all available CPU cores: 8


Running in parallel: 100%|██████████| 10000/10000 [1:53:07<00:00,  1.47it/s]


Total rows: 10000


## Disconnect from Runtime

In [18]:
from google.colab import runtime
runtime.unassign()